In [1]:
# 6-17-2026
# this will take ecoregions and spatially merge similar ones
# similar is characterized by similar env stats stats (mean and std). create total 40-50 regions

In [2]:
import xarray as xr
import numpy as np
import pandas as pd
import regionmask
from tqdm.auto import tqdm

In [3]:
zarr_path = "seasfire_pyromes_ecoregions.zarr"

In [4]:
ds = xr.open_zarr(zarr_path, consolidated=True)

In [5]:
ds

<xarray.Dataset> Size: 76GB
Dimensions:                         (latitude: 720, longitude: 1440, time: 506)
Coordinates:
  * latitude                        (latitude) float64 6kB 89.88 ... -89.88
  * longitude                       (longitude) float64 12kB -179.9 ... 179.9
  * time                            (time) datetime64[ns] 4kB 2011-01-01 ... ...
Data variables: (12/43)
    area                            (latitude, longitude) float32 4MB dask.array<chunksize=(45, 45), meta=np.ndarray>
    biomes                          (latitude, longitude) float32 4MB dask.array<chunksize=(45, 45), meta=np.ndarray>
    cams_co2fire                    (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    cams_frpfire                    (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    drought_code_max                (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    drought_code_mean               (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    ...                              ...
    t2m_max                         (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    t2m_mean                        (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    t2m_min                         (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    tp                              (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    vpd                             (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    ws10                            (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
Attributes:
    crs:          EPSG:4326
    description:  The SeasFire Cube is a scientific datacube for seasonal fir...
    title:        SeasFire Cube: A Global Dataset for Seasonal Fire Modeling ...

In [6]:
print(list(ds.data_vars))

['area', 'biomes', 'cams_co2fire', 'cams_frpfire', 'drought_code_max', 'drought_code_mean', 'ecoregion', 'fcci_ba', 'fcci_ba_valid_mask', 'fcci_fraction_of_burnable_area', 'fcci_fraction_of_observed_area', 'fcci_number_of_patches', 'fwi_max', 'fwi_mean', 'gwis_ba', 'gwis_ba_valid_mask', 'lai', 'lccs_class_1', 'lccs_class_2', 'lccs_class_3', 'lccs_class_4', 'lccs_class_6', 'lccs_class_7', 'lsm', 'lst_day', 'ndvi', 'pop_dens', 'pyrome', 'rel_hum', 'skt', 'ssr', 'ssrd', 'sst', 'swvl1', 'swvl2', 'swvl3', 'swvl4', 't2m_max', 't2m_mean', 't2m_min', 'tp', 'vpd', 'ws10']


In [ ]:
unique_values = np.unique(ds["ecoregion"].values)
num_categories = len(unique_values)
num_categories
# most ecoregions were accounted for

792

c:\Users\Yash\AppData\Local\Programs\Python\Python312\Lib\site-packages\dask\array\numpy_compat.py:58: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)


In [8]:
ds["ecoregion"]

<xarray.DataArray 'ecoregion' (latitude: 720, longitude: 1440)> Size: 8MB
dask.array<open_dataset-ecoregion, shape=(720, 1440), dtype=float64, chunksize=(45, 45), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 6kB 89.88 89.62 89.38 ... -89.38 -89.62 -89.88
  * longitude  (longitude) float64 12kB -179.9 -179.6 -179.4 ... 179.6 179.9
Attributes:
    description:  Spatial mapping of grid cells to the 825 WWF Terrestrial Ec...
    long_name:    WWF Terrestrial Ecoregion ID
    units:        unitless

In [9]:
ecoregion_values = ds["ecoregion"].values.flatten()
ecoregion_values = ecoregion_values[~np.isnan(ecoregion_values)]
unique_ecoregions, counts = np.unique(ecoregion_values, return_counts=True)

In [10]:
df_ecoregions = pd.DataFrame({
    "ccoregion_ID": unique_ecoregions,
    "cell_count": counts
})

In [11]:
df_ecoregions.sort_values(by="cell_count", ascending=False).head(10)

,ccoregion_ID,cell_count
663,80601.0,10686
82,21102.0,10547
784,81327.0,6583
81,21101.0,6376
670,80608.0,6154
673,80611.0,4494
127,30713.0,4089
732,81111.0,4066
391,51115.0,3660
667,80605.0,3542


In [12]:
zero_count = (ds["ecoregion"] == 0).sum()
zero_count

<xarray.DataArray 'ecoregion' ()> Size: 8B
dask.array<sum-aggregate, shape=(), dtype=int64, chunksize=(), chunktype=numpy.ndarray>
Attributes:
    description:  Spatial mapping of grid cells to the 825 WWF Terrestrial Ec...
    long_name:    WWF Terrestrial Ecoregion ID
    units:        unitless

In [13]:
pyrome_counts = ds["pyrome"].stack(points=("latitude", "longitude"))

# unique values and counts
values, counts = np.unique(pyrome_counts, return_counts=True)

for val, count in zip(values, counts):
    print(f"pyrome ID {val}: {count} cells")

pyrome ID 1: 10504 cells
pyrome ID 2: 34500 cells
pyrome ID 3: 25708 cells
pyrome ID 4: 24084 cells
pyrome ID 5: 14412 cells
pyrome ID 255: 927592 cells


In [14]:
ecoregion_coord = ds["ecoregion"].compute()

variables = ["ndvi", "tp", "t2m_mean", "vpd", "fwi_mean"]
stats_list = []

In [ ]:
for var in tqdm(variables, desc="get mean and std per varable"):
    if var in ds:
        grouped = ds[var].groupby(ecoregion_coord)
        
        mean_val = grouped.mean(dim=["time", "stacked_latitude_longitude"]).compute()
        std_val = grouped.std(dim=["time", "stacked_latitude_longitude"]).compute()
        # create df per var
        df_var = pd.DataFrame({
            f"{var}_mean": mean_val.values,
            f"{var}_std": std_val.values
        }, index=mean_val.ecoregion.values)
        
        stats_list.append(df_var)

get mean and std per varable:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
df_features = pd.concat(stats_list, axis=1)

In [ ]:
df_features.head()

,ndvi_mean,ndvi_std,tp_mean,tp_std,t2m_mean_mean,t2m_mean_std,vpd_mean,vpd_std,fwi_mean_mean,fwi_mean_std
10101.0,0.861981,0.018632,87.606903,46.894028,300.059723,0.495307,7.156866,1.218896,0.523723,1.157128
10102.0,0.811376,0.056275,41.696075,44.840942,299.991669,0.916401,8.344715,1.445781,5.379828,7.148313
10103.0,0.831247,0.016678,73.235191,37.511982,300.355103,0.404653,7.995025,0.804152,0.309281,0.589486
10104.0,0.818266,0.037402,75.606750,50.415367,297.252167,1.323315,5.550790,1.390918,1.663618,4.157984
10105.0,0.804765,0.037594,133.072220,83.581337,293.269928,3.177104,4.774403,1.419347,0.165741,0.525632


In [ ]:
raw_unique_ids = np.unique(ds["ecoregion"].values)

raw_unique_ids = raw_unique_ids[~np.isnan(raw_unique_ids)]
raw_unique_ids = [int(x) for x in raw_unique_ids if x != 0]

df_calculated_ids = df_features.index.tolist()

print(f"total unique ecoregions in raw dataset map: {len(raw_unique_ids)}")
print(f"total ecoregions processed in df:   {len(df_calculated_ids)}")
print("-" * 50)
print(f"sample IDs from raw dataset map: {raw_unique_ids[:10]}")
print(f"sample IDs from  dataframe:  {df_calculated_ids[:10]}")

# Automated check to verify complete structural matching
all_match = set(raw_unique_ids) == set(df_calculated_ids)
print(f"\ndataset IDs and df indices match perfectly: {all_match}")

total unique ecoregions in raw dataset map: 791
total ecoregions processed in df:   791
--------------------------------------------------
sample IDs from raw dataset map: [10101, 10102, 10103, 10104, 10105, 10106, 10107, 10108, 10110, 10111]
sample IDs from  dataframe:  [10101.0, 10102.0, 10103.0, 10104.0, 10105.0, 10106.0, 10107.0, 10108.0, 10110.0, 10111.0]

dataset IDs and df indices match perfectly: True


In [ ]:
df_features.index.name = "row_id"
df_features.to_csv("ecoregions_stats.csv")